In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#read the data
DATA_DIR = Path(r"D:\retail_profitability_project\data\raw\SampleSuperstore.csv")

df = pd.read_csv(DATA_DIR)

print (df.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ship Mode     9994 non-null   object 
 1   Segment       9994 non-null   object 
 2   Country       9994 non-null   object 
 3   City          9994 non-null   object 
 4   State         9994 non-null   object 
 5   Postal Code   9994 non-null   int64  
 6   Region        9994 non-null   object 
 7   Category      9994 non-null   object 
 8   Sub-Category  9994 non-null   object 
 9   Sales         9994 non-null   float64
 10  Quantity      9994 non-null   int64  
 11  Discount      9994 non-null   float64
 12  Profit        9994 non-null   float64
dtypes: float64(3), int64(2), object(8)
memory usage: 1015.1+ KB
None


In [2]:
#segment table

segment_table = df.groupby("Segment").agg(
  Total_sales=("Sales", "sum"),
  Average_sales=("Sales", "mean"),
  Total_profit=("Profit", "sum"),
  Average_profit=("Profit", "mean"),
  Total_Discount=("Discount","sum"),
  Average_Discount=("Discount","mean"),
  Number_of_orders=("Quantity", "count")
  
  
).reset_index()

segment_table=segment_table.style.format({
  "Total_sales":"{:,.2f}",
  "Average_sales":"{:,.2f}",
  "Total_profit":"{:,.2f}",
  "Average_profit":"{:,.2f}",
  "Total_Discount":"{:,.2f}",
  "Average_Discount":"{:,.2f}",
})


display(segment_table)

,Segment,Total_sales,Average_sales,Total_profit,Average_profit,Total_Discount,Average_Discount,Number_of_orders
0,Consumer,"1,161,401.34",223.73,"134,119.21",25.84,820.91,0.16,5191
1,Corporate,"706,146.37",233.82,"91,979.13",30.46,477.85,0.16,3020
2,Home Office,"429,653.15",240.97,"60,298.68",33.82,262.33,0.15,1783


In [3]:
# Check for category

df["Category"].value_counts()

Category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64

In [4]:
#Summary statistics for category
category_table = df.groupby("Category").agg(
  Total_sales=("Sales", "sum"),
  Average_sales=("Sales", "mean"),
  Total_profit=("Profit", "sum"),
  Average_profit=("Profit", "mean"),
  Total_Discount=("Discount","sum"),
  Average_Discount=("Discount","mean"),
  Number_of_orders=("Quantity", "count")
  
  
).reset_index()

category_table=category_table.style.format({
  "Total_sales":"{:,.2f}",
  "Average_sales":"{:,.2f}",
  "Total_profit":"{:,.2f}",
  "Average_profit":"{:,.2f}",
  "Total_Discount":"{:,.2f}",
  "Average_Discount":"{:,.2f}",
})

display(category_table)

,Category,Total_sales,Average_sales,Total_profit,Average_profit,Total_Discount,Average_Discount,Number_of_orders
0,Furniture,"741,999.80",349.83,"18,451.27",8.70,368.89,0.17,2121
1,Office Supplies,"719,047.03",119.32,"122,490.80",20.33,947.80,0.16,6026
2,Technology,"836,154.03",452.71,"145,454.95",78.75,244.40,0.13,1847


In [5]:
#create a discount band
max_discount = (
  df.groupby("Category")["Discount"]
    .max()
    .sort_values(ascending=False)
)
display(max_discount)

high_discount = df[df["Discount"] > 0.40]

high_discount_count = (
  high_discount.groupby("Category")
  .size()
  .sort_values(ascending=False)
)

display(high_discount_count)





Category
Office Supplies    0.8
Furniture          0.7
Technology         0.7
Name: Discount, dtype: float64

Category
Office Supplies    680
Furniture          218
Technology          35
dtype: int64

In [6]:
#create a discount band
bins = [-0.01,0,0.1,0.2,0.3,0.4,1]
labels = [
  "No Discount",
  "1-10%",
  "11-20%",
  "21-30%",
  "31-40%",
  "Above 40%"
  ]

df["Discount Bound"] = pd.cut(
  df["Discount"],
  bins=bins,
  labels=labels
)
#calculate profit margin
df["Profit_margin"] = (df["Profit"]/df["Sales"].sum())*100


display(df[["Category","Quantity","Sales","Profit","Profit_margin","Discount", "Discount Bound"]].head(20))

,Category,Quantity,Sales,Profit,Profit_margin,Discount,Discount Bound
0,Furniture,2,261.9600,41.9136,0.001825,0.00,No Discount
1,Furniture,3,731.9400,219.5820,0.009559,0.00,No Discount
2,Office Supplies,2,14.6200,6.8714,0.000299,0.00,No Discount
3,Furniture,5,957.5775,-383.0310,-0.016674,0.45,Above 40%
4,Office Supplies,2,22.3680,2.5164,0.000110,0.20,11-20%
5,Furniture,7,48.8600,14.1694,0.000617,0.00,No Discount
6,Office Supplies,4,7.2800,1.9656,0.000086,0.00,No Discount
7,Technology,6,907.1520,90.7152,0.003949,0.20,11-20%
8,Office Supplies,3,18.5040,5.7825,0.000252,0.20,11-20%
9,Office Supplies,5,114.9000,34.4700,0.001501,0.00,No Discount


In [29]:
# summary statistics by sub-category

sub_category_stats = df.groupby("Sub-Category").agg(
  Total_Revenue=("Sales","sum"),
  Total_Profit=("Profit","sum"),
  Total_Quantity=("Quantity","sum"),
  Average_Revenue=("Sales","mean"),
  Average_Profit=("Profit","mean"),
  Average_Discount=("Discount","mean")
)
sub_category_stats["Profit_Margin"] =(
  sub_category_stats["Total_Profit"]/
  sub_category_stats["Total_Revenue"]
)*100

sub_category_stats = sub_category_stats.sort_values("Profit_Margin", ascending=False)

sub_category_stats = sub_category_stats.style.format({
  "Total_Revenue":"{:,.2f}",
  "Total_Quantity":"{:,.0f}",
  "Total_Profit":"{:,.2f}",
  "Average_Profit":"{:,.2f}",
  "Average_Sales":"{:,.2f}",
  "Average_Discount":"{:,.2f}",
  "Profit_Margin":"{:,.2f}%"
})
display(sub_category_stats)

,Total_Revenue,Total_Profit,Total_Quantity,Average_Revenue,Average_Profit,Average_Discount,Profit_Margin
Sub-Category,,,,,,,
Labels,"12,486.31","5,546.25","1,400",34.303055,15.24,0.07,44.42%
Paper,"78,479.21","34,053.57","5,178",57.284092,24.86,0.07,43.39%
Envelopes,"16,476.40","6,964.18",906,64.867724,27.42,0.08,42.27%
Copiers,"149,528.03","55,617.82",234,2198.941618,817.91,0.16,37.20%
Fasteners,"3,024.28",949.52,914,13.936774,4.38,0.08,31.40%
Accessories,"167,380.32","41,936.64","2,976",215.974604,54.11,0.08,25.05%
Art,"27,118.79","6,527.79","3,000",34.068834,8.20,0.07,24.07%
Appliances,"107,532.16","18,138.01","1,729",230.755710,38.92,0.17,16.87%
Binders,"203,412.73","30,221.76","5,974",133.560560,19.84,0.37,14.86%


In [7]:
#Inspect regions
df["Country"].value_counts()
df["Region"].value_counts()
df["City"].value_counts()
df["State"].value_counts()

State
California              2001
New York                1128
Texas                    985
Pennsylvania             587
Washington               506
Illinois                 492
Ohio                     469
Florida                  383
Michigan                 255
North Carolina           249
Arizona                  224
Virginia                 224
Georgia                  184
Tennessee                183
Colorado                 182
Indiana                  149
Kentucky                 139
Massachusetts            135
New Jersey               130
Oregon                   124
Wisconsin                110
Maryland                 105
Delaware                  96
Minnesota                 89
Connecticut               82
Missouri                  66
Oklahoma                  66
Alabama                   61
Arkansas                  60
Rhode Island              56
Utah                      53
Mississippi               53
South Carolina            42
Louisiana                 42
Nevada  

In [8]:
#summary statistics by  State

state_stats = (
  df.groupby("State")
    .agg(
      Total_Revenue=("Sales", "sum"),
      Total_Quantity=("Quantity","sum"),
      Total_Profit=("Profit","sum"),
      Average_Profit=("Profit","mean"),
      Average_Sales=("Sales","mean"),
      Average_Discount=("Discount","mean")

  )
  
)
state_stats["Profit_Margin"] = (
  state_stats["Total_Profit"]/
  state_stats["Total_Revenue"]
)*100

state_stats = state_stats.sort_values("Profit_Margin", ascending=False)

state_stats = state_stats.style.format({
  "Total_Revenue":"{:,.2f}",
  "Total_Quantity":"{:,.0f}",
  "Total_Profit":"{:,.2f}",
  "Average_Profit":"{:,.2f}",
  "Average_Sales":"{:,.2f}",
  "Average_Discount":"{:,.2f}",
  "Profit_Margin":"{:,.2f}%"
})

display(state_stats)

,Total_Revenue,Total_Quantity,Total_Profit,Average_Profit,Average_Sales,Average_Discount,Profit_Margin
State,,,,,,,
District of Columbia,"2,865.02",40,"1,059.59",105.96,286.50,0.00,36.98%
Delaware,"27,451.07",367,"9,977.37",103.93,285.95,0.01,36.35%
Minnesota,"29,863.15",331,"10,823.19",121.61,335.54,0.00,36.24%
Maine,"1,270.53",35,454.49,56.81,158.82,0.00,35.77%
Arkansas,"11,678.13",240,"4,008.69",66.81,194.64,0.00,34.33%
Indiana,"53,555.36",578,"18,382.94",123.38,359.43,0.00,34.33%
Georgia,"49,095.84",705,"16,250.04",88.32,266.83,0.00,33.10%
Montana,"5,589.35",56,"1,833.33",122.22,372.62,0.07,32.80%
Rhode Island,"22,627.96",199,"7,285.63",130.10,404.07,0.02,32.20%


In [9]:
#Summary statistics by City

city_stats = (
  df.groupby("City")
    .agg(
      Total_Revenue=("Sales", "sum"),
      Total_Quantity=("Quantity","sum"),
      Total_Profit=("Profit","sum"),
      Average_Profit=("Profit","mean"),
      Average_Sales=("Sales","mean"),
      Average_Discount=("Discount","mean")

  )
  
)
city_stats["Profit_Margin"] = (
  city_stats["Total_Profit"]/
  city_stats["Total_Revenue"]
)*100

city_stats = city_stats.sort_values("Profit_Margin", ascending=False)

city_stats = city_stats.style.format({
  "Total_Revenue":"{:,.2f}",
  "Total_Quantity":"{:,.0f}",
  "Total_Profit":"{:,.2f}",
  "Average_Profit":"{:,.2f}",
  "Average_Sales":"{:,.2f}",
  "Average_Discount":"{:,.2f}",
  "Profit_Margin":"{:,.2f}%"
})

display( city_stats)

,Total_Revenue,Total_Quantity,Total_Profit,Average_Profit,Average_Sales,Average_Discount,Profit_Margin
City,,,,,,,
Atlantic City,23.36,2,11.68,11.68,23.36,0.00,50.00%
Grand Island,15.96,2,7.98,7.98,15.96,0.00,50.00%
New Brunswick,14.77,3,7.16,3.58,7.38,0.00,48.45%
Summerville,406.94,10,196.91,98.46,203.47,0.00,48.39%
Holland,137.86,11,66.66,22.22,45.95,0.00,48.35%
Antioch,19.44,3,9.33,9.33,19.44,0.00,48.00%
Lindenhurst,55.48,1,26.63,26.63,55.48,0.00,48.00%
Davis,32.40,5,15.55,15.55,32.40,0.00,48.00%
Royal Oak,35.34,4,16.96,8.48,17.67,0.00,48.00%


In [10]:
#inspect sub-category
df["Sub-Category"].value_counts()

Sub-Category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64